In [17]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression,ElasticNet
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error,root_mean_squared_error,confusion_matrix,roc_auc_score,log_loss
from sklearn.metrics import f1_score,accuracy_score,recall_score,precision_score,classification_report
from tqdm import tqdm
import os
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler,PolynomialFeatures
from sklearn.neighbors import KNeighborsClassifier,KNeighborsRegressor
import datetime as dt

In [18]:
train = pd.read_csv("train.csv")
train

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
594189,594189,Male,0,No,No,57,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Two year,No,Bank transfer (automatic),97.55,5460.70,No
594190,594190,Female,0,No,No,72,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,Two year,No,Bank transfer (automatic),91.95,6782.15,No
594191,594191,Female,0,Yes,No,72,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Credit card (automatic),24.40,1871.90,No
594192,594192,Female,0,No,No,32,Yes,Yes,Fiber optic,No,...,No,No,No,Yes,Month-to-month,Yes,Electronic check,86.00,2847.20,No


In [19]:
le=LabelEncoder() 
train['Churn']=le.fit_transform(train['Churn'])
train['Churn']

0         0
1         0
2         0
3         1
4         1
         ..
594189    0
594190    0
594191    0
594192    0
594193    1
Name: Churn, Length: 594194, dtype: int64

In [20]:
X = train.drop('Churn', axis=1)
y = train['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state=26, stratify= train['Churn'])

In [21]:
from sklearn.compose import make_column_selector,ColumnTransformer
ohe=OneHotEncoder(sparse_output=False,drop="first").set_output(transform="pandas")
trans = ColumnTransformer(
    transformers=[("OHE", ohe,make_column_selector(dtype_include=object))],remainder="passthrough",
    verbose_feature_names_out=False).set_output(transform="pandas")

In [22]:

X_trn_ohe = trans.fit_transform(X_train)
X_tst_ohe = trans.transform(X_test)

scalar=StandardScaler().set_output(transform="pandas")
scalar.fit(X_trn_ohe)
X_trn_scl = scalar.transform(X_trn_ohe)
X_tst_scl = scalar.transform(X_tst_ohe)

In [23]:
ks=[1,2,3,4,5,6,7,8,9,10]
scores = []
for k in tqdm(ks) :
    knn= KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_trn_scl, y_train)
    y_pred_prob = knn.predict_proba(X_tst_scl)
    scores.append([k, log_loss(y_test, y_pred_prob)])
df_scores = pd.DataFrame(scores, columns=['ks', 'log_loss'])
df_scores.sort_values('log_loss', ascending=True)

  0%|                                                                                           | 0/10 [00:00<?, ?it/s]

 10%|████████▏                                                                         | 1/10 [02:16<20:25, 136.22s/it]

 20%|████████████████▍                                                                 | 2/10 [04:32<18:10, 136.37s/it]

 30%|████████████████████████▌                                                         | 3/10 [06:48<15:52, 136.08s/it]

 40%|████████████████████████████████▊                                                 | 4/10 [09:07<13:42, 137.07s/it]

...

 80%|█████████████████████████████████████████████████████████████████▌                | 8/10 [18:17<04:34, 137.05s/it]

 90%|█████████████████████████████████████████████████████████████████████████▊        | 9/10 [20:33<02:16, 136.52s/it]

100%|█████████████████████████████████████████████████████████████████████████████████| 10/10 [22:49<00:00, 136.45s/it]

100%|█████████████████████████████████████████████████████████████████████████████████| 10/10 [22:49<00:00, 136.94s/it]

,ks,log_loss
9,10,0.712649
8,9,0.776273
7,8,0.854036
6,7,0.960970
5,6,1.125389
4,5,1.362330
3,4,1.738347
2,3,2.359019
1,2,3.637391
0,1,7.055303


In [24]:
test = pd.read_csv("test.csv")
test

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,594194,Female,0,Yes,No,72,Yes,Yes,Fiber optic,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Electronic check,115.55,8061.50
1,594195,Female,0,Yes,No,71,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),19.80,1336.50
2,594196,Male,0,No,No,12,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),55.55,633.55
3,594197,Male,0,Yes,Yes,71,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,Two year,No,Credit card (automatic),84.10,6457.15
4,594198,Female,0,No,No,15,Yes,No,Fiber optic,Yes,No,No,No,Yes,Yes,Month-to-month,No,Electronic check,90.35,1233.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
254650,848844,Male,0,Yes,Yes,72,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Credit card (automatic),19.95,1443.65
254651,848845,Male,1,Yes,No,16,Yes,Yes,Fiber optic,No,No,No,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,100.15,1563.50
254652,848846,Male,0,Yes,No,35,Yes,Yes,Fiber optic,Yes,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),105.80,3132.75
254653,848847,Female,0,No,No,25,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,Yes,Credit card (automatic),20.25,511.25


In [25]:
X_ohe = trans.fit_transform(X)
X_scalar=scalar.fit_transform(X_ohe)
bm = KNeighborsClassifier(n_neighbors=10, n_jobs=-1)
bm.fit(X_scalar,y)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",10
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",-1


In [26]:
test = pd.read_csv("test.csv") # Remove index_col=0
test_ohe = trans.transform(test)
test_scl = scalar.transform(test_ohe)

In [27]:
submit = pd.read_csv('Sample_Submission.csv')
# Use the scaled test data here
y_pred_prob = bm.predict_proba(test_scl) 
submit["churn"] = y_pred_prob[:, 1]

In [28]:
submit

,id,Churn,churn
0,594194,0,0.0
1,594195,0,0.0
2,594196,0,0.0
3,594197,0,0.0
4,594198,0,0.5
...,...,...,...
254650,848844,0,0.0
254651,848845,0,1.0
254652,848846,0,0.2
254653,848847,0,0.0


In [29]:
submit.to_csv("sbt_knn_20_Apr.csv",index=False)
submit[submit["churn"]>=0.5]

,id,Churn,churn
4,594198,0,0.5
6,594200,0,0.7
9,594203,0,0.6
13,594207,0,0.9
18,594212,0,0.6
...,...,...,...
254636,848830,0,0.6
254643,848837,0,0.5
254648,848842,0,0.5
254649,848843,0,0.5


In [30]:
# Create a fresh copy from the sample submission
final_submit = pd.read_csv('Sample_Submission.csv')

# Use your model predictions (column 1 is the probability of 'Yes')
# test_scl is your scaled test data from cell [15]
final_submit['Churn'] = bm.predict_proba(test_scl)[:, 1]

# Save it - 'index=False' is mandatory
final_submit.to_csv("knn_submission_v1.csv", index=False)